In [64]:
# !pip install langchain langchain-core langchain-community pydantic duckduckgo-search ddgs

# Built-in Tool - DuckDuckGo Search

In [65]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('who is elon musk ??')

print(results)

/Users/apple/Desktop/GenerativeAI_Foundation/myvenv/lib/python3.13/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Elon Reeve Musk[b] (born June 28, 1971) is a businessman and entrepreneur known for his leadership of Tesla, SpaceX, X, and xAI. Musk has been the wealthiest person in the world since 2021; as of … Elon Musk (born June 28, 1971, Pretoria, South Africa) is a South African –born American entrepreneur who cofounded the electronic payment firm PayPal and formed SpaceX, maker of launch vehicles … Apr 26, 2022 · It seems like not a day goes by without billionaire Elon Musk making headlines - and in the future, we may be calling him a trillionaire. The boss of Tesla, SpaceX, and X (formerly Twitter) is... Jun 5, 2025 · Elon Musk, born on June 28, 1971, in Pretoria, South Africa, is a prominent entrepreneur known for his roles in companies like Tesla, SpaceX, and Neuralink. Recently, he welcomed his 14th... May 29, 2025 · Elon Musk is the founder of SpaceX and Tesla Motors, among other companies. Their success has propelled him to become the richest person in the world, with a net worth greate

In [66]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

duckduckgo_search
A wrapper around DuckDuckGo Search. Useful for when you need to answer questions about current events. Input should be a search query.
{'query': {'description': 'search query to look up', 'title': 'Query', 'type': 'string'}}


# Built-in Tool - Shell Tool

In [67]:
# !pip install langchain-experimental

In [68]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('ls')

print(results)

Executing command:
 ls
Tools_In_Langchain.ipynb
Untitled.ipynb



/Users/apple/Desktop/GenerativeAI_Foundation/myvenv/lib/python3.13/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


In [69]:
# !pip install --upgrade --force-reinstall \
#   "langchain==0.3.27" \
#   "langchain-core>=0.3.72" \
#   "langchain-community>=0.4.0"

# Custom Tools in Langchain

In [108]:
from langchain.tools import tool

In [109]:
# Step 1 - Create a function.

def multiply(a,b):
  """Multiply Two numbers"""
  return a * b

In [110]:
# Step 2 - Add type hints.

def multiply(a: float,b: int) -> int:
  """I want to multiply the two integer numbers."""
  return a * b

## Method 1 - Using Tool decorator

In [111]:
# Step 3 - Add tool Decorator.

@tool
def multiply(a: int,b: int) -> int:
  """Multiply Two numbers"""
  return a * b

In [112]:
result = multiply.invoke({"a": 3.0, "b": 5})

In [113]:
print(result)

15


In [114]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply Two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [115]:
# Will send the schema to our LLM Model.
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply Two numbers', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


## Method 2 - Using Structured Tool

In [129]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

In [130]:
class MultiplyInput(BaseModel):
  a: int = Field(required = True, description="The first number to add")
  b: int = Field(required = True, description="The second number to add")

/Users/apple/Desktop/GenerativeAI_Foundation/myvenv/lib/python3.13/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  serialization_alias: str | None = _Unset,
/Users/apple/Desktop/GenerativeAI_Foundation/myvenv/lib/python3.13/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  serialization_alias: str | None = _Unset,


In [131]:
def multiply_func(a: int, b: int) -> int:
  return a * b

In [132]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name = "multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [134]:
result = multiply_tool.invoke({"a": "3", "b": 3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}


## Method 3 - Using BaseTool Class

In [135]:
from langchain.tools import BaseTool
from typing import Type

In [136]:
from typing_extensions import Required
# Arg Schema Using Pydantic

class MultiplyInput(BaseModel):
  a: int = Field(required = True, description = "The first number to add")
  b: int = Field(required = True, description = "The second number to add")

/Users/apple/Desktop/GenerativeAI_Foundation/myvenv/lib/python3.13/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  serialization_alias: str | None = _Unset,
/Users/apple/Desktop/GenerativeAI_Foundation/myvenv/lib/python3.13/site-packages/pydantic/fields.py:1093: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'required'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  serialization_alias: str | None = _Unset,


In [137]:
class MultiplyTool(BaseTool):
  name: str = 'multiply'
  description: str = "Multiply two numbers"

  args_schema: Type[BaseModel] = MultiplyInput

  def _run(self, a: int, b: int) -> int:
    return a * b

In [139]:
multiply_tool = MultiplyTool()

In [140]:
result = multiply_tool.invoke({"a": 3, "b": 3})

In [141]:
result

9

In [142]:
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to add', 'required': True, 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to add', 'required': True, 'title': 'B', 'type': 'integer'}}
